In [ ]:
from pyspark.sql import Window
from pyspark.sql.functions import *
from pyspark.sql.types import *
import time

# Data Reading

In [ ]:
df = spark.read.format("delta")\
    .load("s3://learn-databricks-project-e2e-1-bronze/products")
    
df.show()

In [ ]:
df = df.drop("_rescued_data")
df.show()

# Functions

In [ ]:
df.createOrReplaceTempView("products")

In [ ]:
%sql

CREATE OR REPLACE FUNCTION learn_e2e_1.bronze.discount_func(p_price DOUBLE)
RETURNS DOUBLE
LANGUAGE SQL
RETURN p_price * 0.90


In [ ]:
%sql

SELECT product_id, price, learn_e2e_1.bronze.discount_func(price) AS discounted_price
FROM products

In [ ]:
df.withColumn("discounted_price", expr("learn_e2e_1.bronze.discount_func(price)")).show()

In [ ]:
%sql

CREATE OR REPLACE FUNCTION learn_e2e_1.bronze.upper_func(p_brand STRING)
RETURNS STRING
LANGUAGE PYTHON
AS
$$
    return p_brand.upper()
$$


In [ ]:
%sql
SELECT product_id, brand, learn_e2e_1.bronze.upper_func(brand) AS upper_brand
FROM products

In [ ]:
df.write.format("delta")\
    .mode("overwrite")\
    .option("path", "s3://learn-databricks-project-e2e-1-silver/products")\
    .save()